# UL PR 1 — Mall Customer Segmentation
**Red & White Skill Education | Unsupervised Learning**

Algorithms: K-Means · Agglomerative Hierarchical · DBSCAN

**Dataset:** [Mall Customer Segmentation Dataset on Kaggle](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python)

Download `Mall_Customers.csv` and place it in the `data/` folder beside this notebook.

## Task 1: Data Loading & Exploratory Data Analysis (EDA)
    **Objective:** Load, inspect and understand the customer data before clustering.
    **Expected output:** Dataset summary, histograms, pairplot, correlation heatmap and written observations.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["axes.titlecolor"] = "#CC0000"
DATA_PATH = Path("data/Mall_Customers.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("Mall_Customers.csv")
df = pd.read_csv(DATA_PATH)
display(df.head())
df.info()
display(df.describe())
print("Shape:", df.shape, "| Null values:", int(df.isnull().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))

: 

In [ ]:
df = df.drop_duplicates().copy()
df = df.rename(columns={
    "Annual Income (k$)": "Annual_Income",
    "Spending Score (1-100)": "Spending_Score"
})
df = df.drop(columns=["CustomerID"])
encoder = LabelEncoder()
df["Gender"] = encoder.fit_transform(df["Gender"])
# Match PR 1 encoding explicitly: Female=0, Male=1.
df["Gender"] = df["Gender"].map({"Female": 0, "Male": 1}).fillna(df["Gender"]).astype(int)
display(df.head())

In [ ]:
for col in ["Age", "Annual_Income", "Spending_Score"]:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(df[col], kde=True, ax=ax, color="#CC0000")
    ax.set_title(f"Distribution of {col}")
    ax.set_xlabel(col); ax.set_ylabel("Count")
    plt.show()

sns.pairplot(df, hue=None)
plt.show()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", ax=ax)
ax.set_title("Feature Correlation Heatmap")
plt.show()

**EDA observations (complete after reviewing your generated plots):**
- Inspect the histograms for the spread and shape of Age, Annual_Income and Spending_Score.
- Use the pairplot to look for visible groupings, especially in Annual_Income versus Spending_Score.
- Use the heatmap values to identify strong positive or negative correlations; do not assume a relationship without checking the displayed values.
- Annual_Income and Spending_Score are useful for a 2D customer-segment view because they represent purchasing capacity and spending behaviour.

## Task 2: Feature Scaling & Feature Selection
    **Objective:** Standardise numeric features and create the required two-feature clustering input.
    **Expected output:** `df_scaled` and `df_2f`.

In [ ]:
scaler = StandardScaler()
numeric_features = ["Age", "Annual_Income", "Spending_Score"]
df_scaled = df.copy()
df_scaled[numeric_features] = scaler.fit_transform(df[numeric_features])
df_2f = df_scaled[["Annual_Income", "Spending_Score"]].copy()
display(df_scaled.head())
display(df_2f.head())

**Why StandardScaler?** Distance-based algorithms such as K-Means and DBSCAN use distances between observations. Annual income, age and spending score have different raw ranges. Scaling prevents a feature with a larger numeric magnitude from dominating distance calculations. Gender is already binary and is not scaled for this assignment. The primary clustering tasks use Annual_Income and Spending_Score for clear 2D visualisation.

## Task 3: K-Means Clustering
    **Objective:** Select a suitable k, fit K-Means and interpret customer groups.
    **Expected output:** Elbow and Silhouette plots, cluster scatter plot with centroids, and profile table.

In [ ]:
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_2f)
    inertias.append(km.inertia_)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 11), inertias, marker="o")
ax.set(title="Elbow Method — Optimal k for K-Means", xlabel="k", ylabel="Inertia")
plt.show()

**Inertia** is the sum of squared distances from each observation to its assigned cluster centroid. It decreases as k increases. The elbow is the point after which adding more clusters gives a smaller improvement. Select and justify k by looking at the curve alongside the Silhouette Score; the assignment notes k=5 is typical, not guaranteed.

In [ ]:
silhouette_values = {}
for k in range(2, 11):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(df_2f)
    silhouette_values[k] = silhouette_score(df_2f, labels)
best_k = max(silhouette_values, key=silhouette_values.get)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(silhouette_values.keys()), list(silhouette_values.values()), marker="o")
ax.set(title="Silhouette Score by k", xlabel="k", ylabel="Silhouette Score")
plt.show()
print("Best k by silhouette:", best_k, "| Scores:", silhouette_values)

In [ ]:
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["KMeans_Cluster"] = kmeans.fit_predict(df_2f)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="Annual_Income", y="Spending_Score", hue="KMeans_Cluster", palette="Set1", ax=ax)
ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], c="black", marker="X", s=200, label="Centroids")
ax.set_title("K-Means Clustering — Annual Income vs Spending Score")
plt.show()
kmeans_profile = df.groupby("KMeans_Cluster")[["Age", "Annual_Income", "Spending_Score"]].mean().round(2)
display(kmeans_profile)

**Business interpretation guide:** Give each cluster a name after inspecting its mean income and spending score. Typical labels include High Income/High Spenders, High Income/Low Spenders, Low Income/High Spenders, Low Income/Low Spenders and Middle Group. Recommend one relevant marketing action for each observed segment.

## Task 4: Agglomerative Hierarchical Clustering
    **Objective:** Use Ward linkage and a dendrogram to examine hierarchical groupings.
    **Expected output:** Dendrogram, hierarchical scatter plot, side-by-side comparison and profile table.

In [ ]:
linked = linkage(df_2f, method="ward")
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(linked, truncate_mode="lastp", p=30, leaf_rotation=90, leaf_font_size=8, ax=ax)
ax.set_title("Dendrogram — Agglomerative Hierarchical Clustering (Ward Linkage)")
ax.set_xlabel("Customer groups"); ax.set_ylabel("Ward distance")
plt.show()

**Ward linkage** merges groups to minimise the increase in within-cluster variance. In a dendrogram, the vertical height represents the distance at which groups merge. A horizontal cut across a large vertical gap is a candidate cluster count; justify the chosen cut from the displayed dendrogram.

In [ ]:
hierarchical_k = best_k
hierarchical = AgglomerativeClustering(n_clusters=hierarchical_k, linkage="ward")
df["Hier_Cluster"] = hierarchical.fit_predict(df_2f)
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x="Annual_Income", y="Spending_Score", hue="Hier_Cluster", palette="Set2", ax=ax)
ax.set_title("Agglomerative Hierarchical Clustering — Annual Income vs Spending Score")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df, x="Annual_Income", y="Spending_Score", hue="KMeans_Cluster", palette="Set1", ax=axes[0], legend=False)
axes[0].set_title("K-Means")
sns.scatterplot(data=df, x="Annual_Income", y="Spending_Score", hue="Hier_Cluster", palette="Set2", ax=axes[1], legend=False)
axes[1].set_title("Agglomerative Hierarchical")
plt.show()
hier_profile = df.groupby("Hier_Cluster")[["Age", "Annual_Income", "Spending_Score"]].mean().round(2)
display(hier_profile)

**Comparison:** K-Means is centroid-based and assigns every observation to the nearest centroid. Agglomerative clustering builds groups by repeatedly merging nearby observations/groups. Border-region assignments may differ because the algorithms use different clustering strategies. Compare the two profile tables before deciding whether segment definitions are consistent.

## Task 5: DBSCAN Clustering
    **Objective:** Estimate eps, test parameter combinations and identify dense regions and noise points.
    **Expected output:** 4-NN distance plot, parameter grid table and DBSCAN scatter plot.

In [ ]:
neighbors = NearestNeighbors(n_neighbors=4)
neighbors_fit = neighbors.fit(df_2f)
distances, indices = neighbors_fit.kneighbors(df_2f)
distances = np.sort(distances[:, -1])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(distances)
ax.set(title="4-NN Distance Plot — Estimating DBSCAN eps", xlabel="Points sorted by distance", ylabel="4th-neighbour distance")
plt.show()

In [ ]:
grid_rows = []
for eps in [0.2, 0.3, 0.4, 0.5, 0.6]:
    for min_samples in [3, 4, 5, 6]:
        labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(df_2f)
        n_clusters = len(set(labels) - {-1})
        noise_count = int(np.sum(labels == -1))
        valid = labels != -1
        score = np.nan
        if n_clusters >= 2 and valid.sum() > n_clusters:
            score = silhouette_score(df_2f.loc[valid], labels[valid])
        grid_rows.append({"eps": eps, "min_samples": min_samples, "n_clusters": n_clusters,
                          "noise_points": noise_count, "silhouette_score": score})
dbscan_grid = pd.DataFrame(grid_rows)
display(dbscan_grid)
candidates = dbscan_grid.dropna(subset=["silhouette_score"])
if len(candidates):
    selected = candidates.sort_values(["silhouette_score", "noise_points"], ascending=[False, True]).iloc[0]
    best_eps, best_min_samples = float(selected["eps"]), int(selected["min_samples"])
else:
    best_eps, best_min_samples = 0.5, 4
print("Selected eps:", best_eps, "| min_samples:", best_min_samples)

In [ ]:
dbscan = DBSCAN(eps=best_eps, min_samples=best_min_samples)
df["DBSCAN_Cluster"] = dbscan.fit_predict(df_2f)
fig, ax = plt.subplots(figsize=(8, 5))
noise_mask = df["DBSCAN_Cluster"] == -1
sns.scatterplot(data=df[~noise_mask], x="Annual_Income", y="Spending_Score",
                hue="DBSCAN_Cluster", palette="Set2", ax=ax)
ax.scatter(df.loc[noise_mask, "Annual_Income"], df.loc[noise_mask, "Spending_Score"],
           c="black", marker="x", label="Noise")
ax.set_title("DBSCAN Clustering — Annual Income vs Spending Score")
plt.show()

**DBSCAN concepts:** `eps` is the radius of a neighbourhood. `min_samples` is the minimum number of points needed to form a dense region. Points that do not belong to a sufficiently dense region are marked as noise (`-1`).

**DBSCAN vs K-Means:** DBSCAN infers clusters from density and does not require k in advance; it can identify irregularly shaped groups and noise. K-Means requires k and assigns every point to a cluster, tending toward compact/spherical groups. Compare the number and stability of segments across all three algorithms using the generated plots and tables.

## Task 6: Algorithm Comparison & Business Insights
    **Objective:** Compare cluster layouts and metrics, then translate segments into marketing recommendations.
    **Expected output:** Three-panel figure, metrics table and business insight report.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title, palette in [
    (axes[0], "KMeans_Cluster", "K-Means", "Set1"),
    (axes[1], "Hier_Cluster", "Hierarchical", "Set2"),
    (axes[2], "DBSCAN_Cluster", "DBSCAN", "Set2")
]:
    sns.scatterplot(data=df, x="Annual_Income", y="Spending_Score", hue=col, palette=palette, ax=ax, legend=False)
    ax.set_title(title)
fig.suptitle("Clustering Algorithm Comparison — Mall Customer Segmentation")
plt.show()

In [ ]:
def safe_metrics(X, labels, exclude_noise=False):
    labels = np.asarray(labels)
    if exclude_noise:
        mask = labels != -1
        X = np.asarray(X)[mask]
        labels = labels[mask]
    n_clusters = len(set(labels))
    if n_clusters < 2 or len(labels) <= n_clusters:
        return n_clusters, np.nan, np.nan, np.nan
    return (n_clusters, silhouette_score(X, labels),
            davies_bouldin_score(X, labels), calinski_harabasz_score(X, labels))

metric_rows = []
for name, col, noise in [
    ("K-Means", "KMeans_Cluster", False),
    ("Agglomerative", "Hier_Cluster", False),
    ("DBSCAN", "DBSCAN_Cluster", True)
]:
    n, sil, dbi, chi = safe_metrics(df_2f, df[col], exclude_noise=noise)
    metric_rows.append({"Algorithm": name, "n_clusters": n, "Silhouette Score": sil,
                        "Davies-Bouldin Index": dbi, "Calinski-Harabasz Index": chi})
metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))

### Business Insights for Mall Management

Use the K-Means profile table as the primary interpretable segmentation view. For each segment found in your results:
- **High income / high spending:** loyalty rewards, premium previews and personalised benefits.
- **High income / low spending:** premium product discovery and targeted incentives.
- **Low income / high spending:** budget bundles, value deals and reward points.
- **Low income / low spending:** low-cost seasonal promotions and entry-level offers.
- **Middle segment:** test category-specific campaigns and measure response.

**Deployment recommendation:** K-Means is a practical choice when management needs simple, explainable customer segments. Choose Hierarchical when the dendrogram hierarchy is important. Choose DBSCAN when unknown cluster shapes and low-density/noise points matter. Base the final decision on your plots, metrics and business objective.

## Task 7: Notebook Quality, Structure & Submission
    **Objective:** Ensure the notebook is readable, reproducible and ready for submission.
    **Expected output:** A clean run, exported HTML, repository files and video link.

### Final submission checklist
- [ ] All seven tasks have Markdown objectives and expected outputs.
- [ ] All plots have descriptive titles and labelled axes.
- [ ] Kernel restarted and **Restart & Run All** completed without errors.
- [ ] Exported as `UL_PR1.html`.
- [ ] GitHub repository contains `UL_PR1.ipynb`, `UL_PR1.html`, `requirements.txt`, `README.md` and `/screenshots`.
- [ ] Screenshots include Elbow, dendrogram, k-distance and three-panel comparison plots.
- [ ] Recorded 5–10 minute face + screen video and added its shareable URL to README.
- [ ] At least three meaningful Git commits.